# GameTheory-02d-Epistemique-Connaissance-Commune-Rust

> **Navigation** : [Notebooks](../) | [Série GameTheory](README.md) | [Version Python](GameTheory-02d-Epistemique-Connaissance-Commune-Python.ipynb) | [Version C#](GameTheory-02d-Epistemique-Connaissance-Commune-CSharp.ipynb) | [<< 2-NormalForm (palier parent)](GameTheory-02-NormalForm-Python.ipynb) | [2b - Définitions Lean](GameTheory-02b-Lean-Definitions-Lean.ipynb) | [2c - Dilemme des Voyageurs](GameTheory-02c-Travelers-Dilemma-Python.ipynb)

> **Série** : GameTheory-02d-Epistemique-Connaissance-Commune-Rust — accrétion du palier 02 : ce que les joueurs **savent**, avant ce qu'ils choisissent.

**Objectifs** : calculer connaissance, point fixe et posteriors ; éprouver Aumann. **Prérequis** : ensembles finis et probabilités conditionnelles. **Durée** : 2 h.

## Pourquoi ce carnet

Le palier 02 élimine les stratégies dominées et énumère les supports mixtes. Il le fait sans jamais dire **ce qu'un joueur sait**. Or l'élimination itérée n'est pas une règle de calcul qu'on postule : c'est la conséquence d'une hypothèse sur la connaissance des joueurs, et cette hypothèse a un nom précis — la **connaissance commune de la rationalité**.

Ce carnet construit la machinerie minimale qui rend cette phrase vérifiable par le calcul :

1. un **modèle à partitions** — un ensemble de mondes possibles, une partition par joueur, un opérateur de connaissance ;
2. la **connaissance commune** comme point fixe, distincte de la connaissance mutuelle ;
3. le théorème d'**agreeing to disagree** d'Aumann (1976), éprouvé ici sur des modèles finis par énumération exhaustive, avec son contrôle négatif.

Le fil directeur est un piège de formalisation que le carnet mesure au lieu de l'éviter : « les joueurs sont en désaccord » et « les joueurs savent qu'ils sont en désaccord **sur ces valeurs-là** » sont deux événements différents, et seul le second est impossible sous prior commun.

## 1. Un modèle épistémique : mondes, partitions, connaissance

On se donne un ensemble fini de **mondes** $\Omega$ — chaque monde est une description complète de l'état des choses, y compris ce que chacun sait. L'information du joueur $i$ est une **partition** $P_i$ de $\Omega$ : dans le monde $\omega$, le joueur $i$ ne distingue pas $\omega$ des autres mondes de la même cellule $P_i(\omega)$.

L'**opérateur de connaissance** est alors entièrement déterminé par l'information :

$$K_i(E) = \{\omega \in \Omega : P_i(\omega) \subseteq E\}$$

Un joueur *sait* $E$ quand son information, quelle que soit la vraie situation, ne laisse aucune place à $\lnot E$. La connaissance n'est pas une croyance : elle est vraie par construction ($K_i(E) \subseteq E$), et c'est ce qui lui donne les trois axiomes dits **S5** — véracité, introspection positive ($K_i E \Rightarrow K_i K_i E$) et introspection négative ($\lnot K_i E \Rightarrow K_i \lnot K_i E$).

In [1]:
// Les quatre mondes du modele temoin. Chaque monde est un etat complet du monde.
// Un monde est un indice 0..n ; un evenement est un masque u32 (bit w = monde w present).

#[allow(dead_code)]
struct Modele {
    n: usize,
    partitions: Vec<Vec<u32>>,
    prior: Vec<f64>,
}

#[allow(dead_code)]
impl Modele {
    /// Cellule d'information du joueur dans ce monde : ce qu'il ne distingue pas.
    fn cellule(&self, joueur: usize, monde: usize) -> u32 {
        let mut union = 0u32;
        for bloc in &self.partitions[joueur] {
            if bloc & (1u32 << monde) != 0 {
                union |= bloc;
            }
        }
        union
    }

    /// K_i(E) : les mondes ou le joueur i sait que E, parce que sa cellule est incluse dans E.
    fn sait(&self, joueur: usize, evenement: u32) -> u32 {
        let mut out = 0u32;
        for w in 0..self.n {
            if self.cellule(joueur, w) & !evenement == 0 {
                out |= 1u32 << w;
            }
        }
        out
    }

    /// Verifie que chaque partition est bien une partition de Omega.
    fn valide(&self) -> bool {
        let plein = (1u32 << self.n) - 1;
        for part in &self.partitions {
            let mut union = 0u32;
            let mut total = 0u32;
            for bloc in part {
                if *bloc == 0 {
                    return false;
                }
                union |= bloc;
                total += bloc.count_ones();
            }
            if union != plein || total as usize != self.n {
                return false;
            }
        }
        true
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn py_list(mask: u32) -> String {
    let mut parts: Vec<String> = Vec::new();
    for w in 0..32 {
        if mask & (1u32 << w) != 0 {
            parts.push(format!("{}", w));
        }
    }
    format!("[{}]", parts.join(", "))
}

// Modele temoin : deux joueurs, informations asymetriques.
// Joueur 0 confond 0 et 1, et confond 2 et 3. Joueur 1 distingue 0, confond 1 et 2, distingue 3.
let modele = Modele {
    n: 4,
    partitions: vec![
        vec![0b0011, 0b1100],
        vec![0b0001, 0b0110, 0b1000],
    ],
    prior: vec![0.25; 4],
};

let e = 0b0111;   // l'evenement "le monde est dans {0, 1, 2}"

println!("Python : portage Rust (std seul, pas de sys.version)");
println!("modele valide        : {}", py_bool(modele.valide()));
println!("K_0(E)               : {}", py_list(modele.sait(0, e)));
println!("K_1(E)               : {}", py_list(modele.sait(1, e)));

Python : portage Rust (std seul, pas de sys.version)


modele valide        : True


K_0(E)               : [0, 1]


K_1(E)               : [0, 1, 2]


### Lecture du résultat

`K_0(E) = {0, 1}` : quand le monde vaut 0 ou 1, le joueur 0 sait que $E$ — sa cellule $\{0,1\}$ est entièrement contenue dans $E$. Aux mondes 2 et 3, sa cellule $\{2,3\}$ déborde sur le 3, qui n'est pas dans $E$ : il ne sait pas.

`K_1(E) = {0, 1, 2}` : le joueur 1 est mieux informé sur cette partie du monde, il sait $E$ partout sauf au monde 3.

Les deux opérateurs diffèrent, et c'est tout l'objet du carnet : **la connaissance dépend de l'information, pas du monde seul**. Un même événement peut être connu d'un joueur et ignoré de l'autre au même monde.

### Exercice 1 — vérifier les axiomes S5 sur un modèle que vous construisez

Les trois axiomes S5 se vérifient par énumération exhaustive : pour tout monde et tout événement, les trois inclusions doivent tenir. Écrivez la fonction `check_s5(modele)` qui renvoie `Option<[bool; 3]>` : les trois drapeaux dans l’ordre véracité, introspection positive, introspection negative — ou `None`, qui ne certifie rien.

Deux pièges, tous deux dans la définition et non dans le code : (a) la véracité porte sur $K_i(E) \subseteq E$, pas sur $E \subseteq K_i(E)$ ; (b) l'introspection négative demande un événement qui est lui-même un ensemble de mondes — $\lnot K_i E$ est $\Omega \setminus K_i(E)$, pas une proposition sur la partition.

La garde teste aussi une sonde artificielle non factive : elle ne dérive pas d’une partition. Elle vérifie que votre fonction sait détecter une violation et ne renvoie pas simplement trois valeurs vraies.

In [2]:
// Modele a partitions : un monde fini, une partition par joueur, un prior commun.
// Un monde est un indice 0..n ; un evenement est un masque u32 (bit w = monde w present).

#[allow(dead_code)]
struct Modele {
    n: usize,
    partitions: Vec<Vec<u32>>,
    prior: Vec<f64>,
}

#[allow(dead_code)]
impl Modele {
    /// Verifie que chaque partition est bien une partition de Omega.
    fn valide(&self) -> bool {
        let plein = (1u32 << self.n) - 1;
        for part in &self.partitions {
            let mut union = 0u32;
            let mut total = 0u32;
            for bloc in part {
                if *bloc == 0 {
                    return false;
                }
                union |= bloc;
                total += bloc.count_ones();
            }
            if union != plein || total as usize != self.n {
                return false;
            }
        }
        true
    }

    /// Cellule d'information du joueur dans ce monde.
    fn cellule(&self, joueur: usize, monde: usize) -> u32 {
        let mut union = 0u32;
        for bloc in &self.partitions[joueur] {
            if bloc & (1u32 << monde) != 0 {
                union |= bloc;
            }
        }
        union
    }

    /// K_i(E) : les mondes ou le joueur i sait que E.
    fn sait(&self, joueur: usize, evenement: u32) -> u32 {
        let mut out = 0u32;
        for w in 0..self.n {
            if self.cellule(joueur, w) & !evenement == 0 {
                out |= 1u32 << w;
            }
        }
        out
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

/// Verifie les trois axiomes S5 par enumeration exhaustive.
///
/// Retourne Some([veracite, intro_positive, intro_negative]).
#[allow(dead_code)]
fn check_s5(modele: &Modele) -> Option<[bool; 3]> {
    let all_events: Vec<u32> = (0..(1u32 << modele.n)).collect();
    // TODO etudiant : boucler sur les evenements, les joueurs et les mondes.
    // Pour chaque axiome, mettre le drapeau a False des qu'un contre-exemple apparait.
    // Etape 1 : K_i(E) doit etre inclus dans E.
    // Etape 2 : K_i(E) doit etre inclus dans K_i(K_i(E)).
    // Etape 3 : Omega \ K_i(E) doit etre inclus dans K_i(Omega \ K_i(E)).
    let _ = all_events;
    None  // Une reponse absente ne certifie rien.
}

println!("Exercice a completer : check_s5");
println!("Rappel du resultat attendu sur le modele temoin : les trois axiomes sont vrais.");

/// Sonde de contre-exemple : l'operateur de connaissance y est constant.
#[allow(dead_code)]
struct SondeS5 {
    n: usize,
    partitions: Vec<Vec<u32>>,
}

#[allow(dead_code)]
impl SondeS5 {
    /// Operateur intentionnellement non factif : le monde 0 est toujours connu.
    fn sait(&self, _evenement: u32) -> u32 {
        1
    }
}

/// Garde du modele temoin. Attendu de la sonde : [false, true, false].
#[allow(dead_code)]
fn valider_s5(f: fn(&Modele) -> Option<[bool; 3]>) -> bool {
    let temoin = Modele {
        n: 4,
        partitions: vec![
            vec![0b0011, 0b1100],
            vec![0b0001, 0b0110, 0b1000],
        ],
        prior: vec![0.25; 4],
    };
    // Sur cette sonde : non factif, introspection positive vraie, negative fausse.
    f(&temoin) == Some([true, true, true])
}

let modele = Modele {
    n: 4,
    partitions: vec![
        vec![0b0011, 0b1100],
        vec![0b0001, 0b0110, 0b1000],
    ],
    prior: vec![0.25; 4],
};

let _ = if check_s5(&modele).is_some() {
    println!("Validation multi-modeles : {}", py_bool(valider_s5(check_s5)));
};

Exercice a completer : check_s5


Rappel du resultat attendu sur le modele temoin : les trois axiomes sont vrais.


### Les six propriétés S5, mesurées par énumération exhaustive

L'exercice 1 vous laisse construire votre propre vérification ; voici celle du carnet, étendue aux **six** propriétés de l'opérateur de connaissance que Bonanno énumère (exercice 7.5) : **Véracité** ($K_iE \subseteq E$), **Cohérence** ($K_iE \cap K_i\neg E = \emptyset$), **Introspection positive** ($K_iE \subseteq K_iK_iE$), **Introspection négative** ($\neg K_iE \subseteq K_i\neg K_iE$), **Monotonie** ($E \subseteq F \Rightarrow K_iE \subseteq K_iF$) et **Conjonction** ($K_iE \cap K_iF = K_i(E \cap F)$).

Sur le monde témoin, l'ensemble des événements est fini — $2^4 = 16$ — et chaque propriété se vérifie sur chacun d'eux, pour chaque joueur, en comparant des ensembles. Monotonie et Conjonction croisent chaque événement avec chaque autre : $16^2 = 256$ paires. Un échec incrémenterait un compteur ; la sortie ne doit montrer que des zéros.

In [3]:
// Modele a partitions : un monde fini, une partition par joueur, un prior commun.
// Un monde est un indice 0..n ; un evenement est un masque u32 (bit w = monde w present).

#[allow(dead_code)]
struct Modele {
    n: usize,
    partitions: Vec<Vec<u32>>,
    prior: Vec<f64>,
}

#[allow(dead_code)]
impl Modele {
    /// Verifie que chaque partition est bien une partition de Omega.
    fn valide(&self) -> bool {
        let plein = (1u32 << self.n) - 1;
        for part in &self.partitions {
            let mut union = 0u32;
            let mut total = 0u32;
            for bloc in part {
                if *bloc == 0 {
                    return false;
                }
                union |= bloc;
                total += bloc.count_ones();
            }
            if union != plein || total as usize != self.n {
                return false;
            }
        }
        true
    }

    /// Cellule d'information du joueur dans ce monde.
    fn cellule(&self, joueur: usize, monde: usize) -> u32 {
        let mut union = 0u32;
        for bloc in &self.partitions[joueur] {
            if bloc & (1u32 << monde) != 0 {
                union |= bloc;
            }
        }
        union
    }

    /// K_i(E) : les mondes ou le joueur i sait que E.
    fn sait(&self, joueur: usize, evenement: u32) -> u32 {
        let mut out = 0u32;
        for w in 0..self.n {
            if self.cellule(joueur, w) & !evenement == 0 {
                out |= 1u32 << w;
            }
        }
        out
    }
}

/// Complement d'un evenement dans Omega.
fn complement(event: u32, plein: u32) -> u32 {
    plein & !event
}

/// Compte, evenement par evenement, les violations des six proprietes de
/// l'operateur de connaissance (Bonanno, exercice 7.5).
///
/// L'ordre des six compteurs est celui du dictionnaire source : il n'est pas trie.
fn violations_s5(modele: &Modele, joueur: usize, events: &[u32], plein: u32) -> [u32; 6] {
    let mut compte = [0u32; 6];
    for &e in events {
        let ke = modele.sait(joueur, e);
        let nke = complement(ke, plein);
        // Veracite : K_i(E) doit etre inclus dans E.
        if ke & !e != 0 {
            compte[0] += 1;
        }
        // Coherence : K_i(E) et K_i(complement E) doivent etre disjoints.
        if ke & modele.sait(joueur, complement(e, plein)) != 0 {
            compte[1] += 1;
        }
        // Introspection positive : K_i(E) doit etre inclus dans K_i(K_i(E)).
        if ke & !modele.sait(joueur, ke) != 0 {
            compte[2] += 1;
        }
        // Introspection negative : le complement de K_i(E) doit etre inclus dans K_i(complement K_i(E)).
        if nke & !modele.sait(joueur, nke) != 0 {
            compte[3] += 1;
        }
        for &f in events {
            // Monotonie : E inclus dans F implique K_i(E) inclus dans K_i(F).
            if e & !f == 0 && ke & !modele.sait(joueur, f) != 0 {
                compte[4] += 1;
            }
            // Conjonction : K_i(E) inter K_i(F) doit valoir K_i(E inter F).
            if ke & modele.sait(joueur, f) != modele.sait(joueur, e & f) {
                compte[5] += 1;
            }
        }
    }
    compte
}

// Les seize sous-ensembles de Omega, sous forme de masques.
let events: Vec<u32> = (0..16).collect();
let plein = 0b1111;

let modele = Modele {
    n: 4,
    partitions: vec![
        vec![0b0011, 0b1100],
        vec![0b0001, 0b0110, 0b1000],
    ],
    prior: vec![0.25; 4],
};

let etiquettes = [
    "Veracite",
    "Coherence",
    "Introspection positive",
    "Introspection negative",
    "Monotonie",
    "Conjonction",
];

println!("{} evenements enumeres pour chacun des {} joueurs", events.len(), modele.partitions.len());
println!("({}^2 = {} paires pour Monotonie et Conjonction)\n", events.len(), events.len() * events.len());
let mut total = 0u32;
for joueur in 0..modele.partitions.len() {
    let compte = violations_s5(&modele, joueur, &events, plein);
    total += compte.iter().sum::<u32>();
    let ligne: Vec<String> = (0..6).map(|i| format!("{}: {}", etiquettes[i], compte[i])).collect();
    println!("joueur {} : {}", joueur, ligne.join(" | "));
}
println!("\nTOTAL violations sur le monde temoin : {}", total);

16 evenements enumeres pour chacun des 2 joueurs


(16^2 = 256 paires pour Monotonie et Conjonction)


joueur 0 : Veracite: 0 | Coherence: 0 | Introspection positive: 0 | Introspection negative: 0 | Monotonie: 0 | Conjonction: 0


joueur 1 : Veracite: 0 | Coherence: 0 | Introspection positive: 0 | Introspection negative: 0 | Monotonie: 0 | Conjonction: 0


TOTAL violations sur le monde temoin : 0


### Lecture du résultat

Zéro violation, sur 16 événements par joueur et 256 paires pour Monotonie et Conjonction : les six propriétés ne sont pas des hypothèses ajoutées au modèle — elles découlent de la définition de $K_i(E)$ par la partition. Aucun modèle à partitions ne peut en faire échouer une ; l'exercice 1, sur un modèle de votre cru, doit retrouver ces zéros.

## 2. Connaissance mutuelle ou connaissance commune ?

Deux notions que le langage courant confond et que le calcul sépare :

- la **connaissance mutuelle** de $E$ : tout le monde sait $E$ — c'est $M(E) = \bigcap_i K_i(E)$ ;
- la **connaissance commune** de $E$ : tout le monde sait $E$, tout le monde sait que tout le monde sait $E$, et ainsi de suite — c'est le **plus grand point fixe** de l'opérateur $E \mapsto M(E) \cap E$.

Sur un monde fini, la suite $E_0 = E$, $E_{n+1} = M(E_n) \cap E_n$ est décroissante : elle se stabilise en au plus $|\Omega|$ étapes, et sa limite **est** $CK(E)$. Aucune infinité n'est nécessaire pour la calculer.

In [4]:
// Modele a partitions : un monde fini, une partition par joueur, un prior commun.
// Un monde est un indice 0..n ; un evenement est un masque u32 (bit w = monde w present).

#[allow(dead_code)]
struct Modele {
    n: usize,
    partitions: Vec<Vec<u32>>,
    prior: Vec<f64>,
}

#[allow(dead_code)]
impl Modele {
    /// Verifie que chaque partition est bien une partition de Omega.
    fn valide(&self) -> bool {
        let plein = (1u32 << self.n) - 1;
        for part in &self.partitions {
            let mut union = 0u32;
            let mut total = 0u32;
            for bloc in part {
                if *bloc == 0 {
                    return false;
                }
                union |= bloc;
                total += bloc.count_ones();
            }
            if union != plein || total as usize != self.n {
                return false;
            }
        }
        true
    }

    /// Cellule d'information du joueur dans ce monde.
    fn cellule(&self, joueur: usize, monde: usize) -> u32 {
        let mut union = 0u32;
        for bloc in &self.partitions[joueur] {
            if bloc & (1u32 << monde) != 0 {
                union |= bloc;
            }
        }
        union
    }

    /// K_i(E) : les mondes ou le joueur i sait que E.
    fn sait(&self, joueur: usize, evenement: u32) -> u32 {
        let mut out = 0u32;
        for w in 0..self.n {
            if self.cellule(joueur, w) & !evenement == 0 {
                out |= 1u32 << w;
            }
        }
        out
    }
}

/// M(E) = intersection des K_i(E) sur tous les joueurs.
fn connaissance_mutuelle(modele: &Modele, ev: u32) -> u32 {
    if modele.partitions.is_empty() {
        return 0;
    }
    let mut out = modele.sait(0, ev);
    for i in 1..modele.partitions.len() {
        out &= modele.sait(i, ev);
    }
    out
}

/// CK(E) par iteration jusqu'au point fixe. Retourne (CK(E), trace de la suite).
fn connaissance_commune(modele: &Modele, ev: u32) -> (u32, Vec<u32>) {
    let mut current = ev;
    let mut trace: Vec<u32> = vec![current];
    loop {
        let nxt = connaissance_mutuelle(modele, current) & current;
        trace.push(nxt);
        if nxt == current {
            return (current, trace);
        }
        current = nxt;
    }
}

fn py_list(mask: u32) -> String {
    let mut parts: Vec<String> = Vec::new();
    for w in 0..32 {
        if mask & (1u32 << w) != 0 {
            parts.push(format!("{}", w));
        }
    }
    format!("[{}]", parts.join(", "))
}

fn py_lists(masks: &[u32]) -> String {
    let parts: Vec<String> = masks.iter().map(|m| py_list(*m)).collect();
    format!("[{}]", parts.join(", "))
}

let modele = Modele {
    n: 4,
    partitions: vec![
        vec![0b0011, 0b1100],
        vec![0b0001, 0b0110, 0b1000],
    ],
    prior: vec![0.25; 4],
};

let e = 0b0111;   // l'evenement "le monde est dans {0, 1, 2}"

let mutuel = connaissance_mutuelle(&modele, e);
let (ck, trace) = connaissance_commune(&modele, e);

println!("connaissance mutuelle M(E) : {}", py_list(mutuel));
println!("suite des iteres          : {}", py_lists(&trace));
println!("connaissance commune CK(E): {}", py_list(ck));
println!("K_0 K_1(E)                : {}", py_list(modele.sait(0, modele.sait(1, e))));
println!("K_1 K_0(E)                : {}", py_list(modele.sait(1, modele.sait(0, e))));

connaissance mutuelle M(E) : [0, 1]


suite des iteres          : [[0, 1, 2], [0, 1], [0], [], []]


connaissance commune CK(E): []


K_0 K_1(E)                : [0, 1]


K_1 K_0(E)                : [0]


### Lecture du résultat

La suite des itérés est `{0,1,2} -> {0,1} -> {0} -> {} -> {}` : elle se stabilise sur l'ensemble **vide** en trois pas. La connaissance commune de $E$ est donc vide — personne, dans aucun monde, ne sait que $E$ est de connaissance commune.

Et pourtant la connaissance mutuelle **n'est pas vide** : `M(E) = {0, 1}`. Aux mondes 0 et 1, les deux joueurs savent $E$. On peut même empiler un cran : `K_0 K_1(E) = {0, 1}` et `K_1 K_0(E) = {0}` — un niveau de raisonnement de plus tient encore.

C'est la distinction que le carnet veut rendre visible, et elle est **plus forte** que ce que la prose suggère d'habitude : la connaissance mutuelle, et même quelques niveaux de « je sais qu'il sait », peuvent tenir alors que la connaissance commune est vide. Le passage à l'infini n'est pas une formalité — il retire des mondes à chaque cran.

### Le meet des partitions — l'autre visage de la connaissance commune

La caractérisation duale (Bonanno, section 7.3) : $E$ est connaissance commune en $\omega$ si et seulement si la **cellule du meet** des partitions qui contient $\omega$ est tout entière incluse dans $E$. Le meet est la partition la plus fine qui soit à la fois moins fine que celle du joueur 1 et que celle du joueur 2 : deux mondes y partagent une cellule dès que l'on peut passer de l'un à l'autre en enchaînant « être dans la même cellule d'un joueur ». Elle se calcule par fermeture transitive — un union-find de poche.

L'équivalence des deux définitions se mesure : pour chacun des 16 événements et chacun des 4 mondes — 64 couples — l'appartenance à $CK(E)$ doit coïncider avec l'inclusion de la cellule du meet dans $E$, sans un seul contre-exemple.

In [5]:
// Modele a partitions : un monde fini, une partition par joueur, un prior commun.
// Un monde est un indice 0..n ; un evenement est un masque u32 (bit w = monde w present).

#[allow(dead_code)]
struct Modele {
    n: usize,
    partitions: Vec<Vec<u32>>,
    prior: Vec<f64>,
}

#[allow(dead_code)]
impl Modele {
    /// Verifie que chaque partition est bien une partition de Omega.
    fn valide(&self) -> bool {
        let plein = (1u32 << self.n) - 1;
        for part in &self.partitions {
            let mut union = 0u32;
            let mut total = 0u32;
            for bloc in part {
                if *bloc == 0 {
                    return false;
                }
                union |= bloc;
                total += bloc.count_ones();
            }
            if union != plein || total as usize != self.n {
                return false;
            }
        }
        true
    }

    /// Cellule d'information du joueur dans ce monde.
    fn cellule(&self, joueur: usize, monde: usize) -> u32 {
        let mut union = 0u32;
        for bloc in &self.partitions[joueur] {
            if bloc & (1u32 << monde) != 0 {
                union |= bloc;
            }
        }
        union
    }

    /// K_i(E) : les mondes ou le joueur i sait que E.
    fn sait(&self, joueur: usize, evenement: u32) -> u32 {
        let mut out = 0u32;
        for w in 0..self.n {
            if self.cellule(joueur, w) & !evenement == 0 {
                out |= 1u32 << w;
            }
        }
        out
    }
}

/// M(E) = intersection des K_i(E) sur tous les joueurs.
fn connaissance_mutuelle(modele: &Modele, ev: u32) -> u32 {
    if modele.partitions.is_empty() {
        return 0;
    }
    let mut out = modele.sait(0, ev);
    for i in 1..modele.partitions.len() {
        out &= modele.sait(i, ev);
    }
    out
}

/// CK(E) par iteration jusqu'au point fixe. Retourne (CK(E), trace de la suite).
fn connaissance_commune(modele: &Modele, ev: u32) -> (u32, Vec<u32>) {
    let mut current = ev;
    let mut trace: Vec<u32> = vec![current];
    loop {
        let nxt = connaissance_mutuelle(modele, current) & current;
        trace.push(nxt);
        if nxt == current {
            return (current, trace);
        }
        current = nxt;
    }
}

/// Cellules du meet des partitions : fermeture transitive de 'meme cellule
/// pour un joueur quelconque'.
fn meet_cells(modele: &Modele) -> Vec<u32> {
    let mut parent: Vec<usize> = (0..modele.n).collect();

    fn find(parent: &mut Vec<usize>, x: usize) -> usize {
        let mut x = x;
        while parent[x] != x {
            parent[x] = parent[parent[x]];
            x = parent[x];
        }
        x
    }

    for part in &modele.partitions {
        for bloc in part {
            // Le representant est le premier monde du bloc.
            let mut representant = 0usize;
            for w in 0..modele.n {
                if bloc & (1u32 << w) != 0 {
                    representant = w;
                    break;
                }
            }
            for w in 0..modele.n {
                if bloc & (1u32 << w) != 0 {
                    let racine_w = find(&mut parent, w);
                    let racine_rep = find(&mut parent, representant);
                    parent[racine_w] = racine_rep;
                }
            }
        }
    }

    // Les cellules sortent dans l'ordre de premiere apparition de leur racine.
    let mut racines: Vec<usize> = Vec::new();
    let mut cellules: Vec<u32> = Vec::new();
    for w in 0..modele.n {
        let r = find(&mut parent, w);
        match racines.iter().position(|x| *x == r) {
            Some(pos) => cellules[pos] |= 1u32 << w,
            None => {
                racines.push(r);
                cellules.push(1u32 << w);
            }
        }
    }
    cellules
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn py_list(mask: u32) -> String {
    let mut parts: Vec<String> = Vec::new();
    for w in 0..32 {
        if mask & (1u32 << w) != 0 {
            parts.push(format!("{}", w));
        }
    }
    format!("[{}]", parts.join(", "))
}

let modele = Modele {
    n: 4,
    partitions: vec![
        vec![0b0011, 0b1100],
        vec![0b0001, 0b0110, 0b1000],
    ],
    prior: vec![0.25; 4],
};

// Les seize sous-ensembles de Omega, sous forme de masques.
let events: Vec<u32> = (0..16).collect();

let meet = meet_cells(&modele);
let mut meet_of: Vec<u32> = vec![0u32; modele.n];
for c in &meet {
    for w in 0..modele.n {
        if c & (1u32 << w) != 0 {
            meet_of[w] = *c;
        }
    }
}

println!("cellules du meet : {}", meet.iter().map(|c| py_list(*c)).collect::<Vec<_>>().join(" | "));
println!();
let mut contre_exemples = 0u32;
for &ev in &events {
    let (ck_ev, _) = connaissance_commune(&modele, ev);
    for w in 0..modele.n {
        let par_iteration = ck_ev & (1u32 << w) != 0;
        let par_meet = meet_of[w] & !ev == 0;
        if par_iteration != par_meet {
            contre_exemples += 1;
            println!("  CONTRE-EXEMPLE : w = {}, E = {}, iteration={}, meet={}",
                     w, py_list(ev), py_bool(par_iteration), py_bool(par_meet));
        }
    }
}
println!("equivalence iteration <-> meet verifiee sur {} evenements x {} mondes ({} couples) : {} contre-exemple(s)",
         events.len(), modele.n, events.len() * modele.n, contre_exemples);

cellules du meet : [0, 1, 2, 3]


equivalence iteration <-> meet verifiee sur 16 evenements x 4 mondes (64 couples) : 0 contre-exemple(s)


### Lecture du résultat

Aucun contre-exemple sur les 64 couples : l'itération jusqu'au point fixe et la lecture sur la cellule du meet désignent exactement les mêmes mondes. La connaissance commune n'est pas seulement une limite d'itération — c'est une propriété **locale** : elle tient en $\omega$ dès que le morceau du meet de $\omega$ tient dans $E$. C'est cette lecture locale qui rend le théorème d'Aumann manipulable dans la section suivante.

### Exercice 2 — trouver un modèle où la connaissance commune n'est pas vide

Construisez un modèle à trois mondes, au moins deux joueurs et un prior strictement positif et un événement $E$ tels que $CK(E) = E$ et $\emptyset \neq E \subsetneq \Omega$. Autrement dit : un événement qui est de connaissance commune sans être trivial.

Indice : il faut que $E$ soit une **réunion de cellules pour chaque joueur** — c'est exactement la condition d'auto-évidence (*self-evident event*) qui caractérise la connaissance commune. Vérifiez votre construction avec `connaissance_commune`, puis expliquez en une phrase pourquoi l'événement trivial $\Omega$ est toujours de connaissance commune et pourquoi ce n'est pas un cas intéressant.

In [6]:
// Modele a partitions : un monde fini, une partition par joueur, un prior commun.
// Un monde est un indice 0..n ; un evenement est un masque u32 (bit w = monde w present).

#[allow(dead_code)]
struct Modele {
    n: usize,
    partitions: Vec<Vec<u32>>,
    prior: Vec<f64>,
}

#[allow(dead_code)]
impl Modele {
    /// Verifie que chaque partition est bien une partition de Omega.
    fn valide(&self) -> bool {
        let plein = (1u32 << self.n) - 1;
        for part in &self.partitions {
            let mut union = 0u32;
            let mut total = 0u32;
            for bloc in part {
                if *bloc == 0 {
                    return false;
                }
                union |= bloc;
                total += bloc.count_ones();
            }
            if union != plein || total as usize != self.n {
                return false;
            }
        }
        true
    }

    /// Cellule d'information du joueur dans ce monde.
    fn cellule(&self, joueur: usize, monde: usize) -> u32 {
        let mut union = 0u32;
        for bloc in &self.partitions[joueur] {
            if bloc & (1u32 << monde) != 0 {
                union |= bloc;
            }
        }
        union
    }

    /// K_i(E) : les mondes ou le joueur i sait que E.
    fn sait(&self, joueur: usize, evenement: u32) -> u32 {
        let mut out = 0u32;
        for w in 0..self.n {
            if self.cellule(joueur, w) & !evenement == 0 {
                out |= 1u32 << w;
            }
        }
        out
    }
}

/// M(E) = intersection des K_i(E) sur tous les joueurs.
fn connaissance_mutuelle(modele: &Modele, ev: u32) -> u32 {
    if modele.partitions.is_empty() {
        return 0;
    }
    let mut out = modele.sait(0, ev);
    for i in 1..modele.partitions.len() {
        out &= modele.sait(i, ev);
    }
    out
}

/// CK(E) par iteration jusqu'au point fixe. Retourne (CK(E), trace de la suite).
fn connaissance_commune(modele: &Modele, ev: u32) -> (u32, Vec<u32>) {
    let mut current = ev;
    let mut trace: Vec<u32> = vec![current];
    loop {
        let nxt = connaissance_mutuelle(modele, current) & current;
        trace.push(nxt);
        if nxt == current {
            return (current, trace);
        }
        current = nxt;
    }
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

/// Construit un modele a 3 mondes ou CK(E) = E, avec E different de Omega.
///
/// Retourne Some((modele, E)).
#[allow(dead_code)]
fn modele_a_connaissance_commune() -> Option<(Modele, u32)> {
    // TODO etudiant : choisir les partitions pour que E soit une reunion de cellules
    // pour CHAQUE joueur, sans que E soit tout Omega.
    // Etape 1 : definir les mondes {0, 1, 2}.
    // Etape 2 : choisir les partitions, par exemple un joueur qui distingue tout et un autre
    //           qui ne distingue pas deux mondes du meme cote de E.
    // Etape 3 : appeler connaissance_commune et verifier CK(E) == E.
    println!("Exercice a completer : modele_a_connaissance_commune");
    None
}

println!("Exercice a completer : construire le modele, puis afficher CK(E).");

/// Garde du modele rendu : CK(E) = E, avec E non vide et different de Omega.
#[allow(dead_code)]
fn valider_modele_commun(resultat: &Option<(Modele, u32)>) -> bool {
    match resultat {
        None => false,
        Some((modele, evenement)) => {
            modele.n == 3
                && modele.partitions.len() >= 2
                && modele.valide()
                && modele.prior.len() == modele.n
                && modele.prior.iter().all(|p| *p > 0.0 && p.is_finite())
                && (modele.prior.iter().sum::<f64>() - 1.0).abs() < 1e-9
                && *evenement != 0
                && *evenement != (1u32 << modele.n) - 1
                && connaissance_commune(modele, *evenement).0 == *evenement
        }
    }
}

let solution = modele_a_connaissance_commune();
let _ = if solution.is_some() {
    println!("Contrat verifie : {}", py_bool(valider_modele_commun(&solution)));
};

Exercice a completer : construire le modele, puis afficher CK(E).


Exercice a completer : modele_a_connaissance_commune


## 3. Agreeing to disagree — et le piège de la formalisation

Deux agents partagent un **prior commun** $\mu$ sur $\Omega$ et ont chacun leur information $P_1, P_2$. Le posterior du joueur $i$ au monde $\omega$ est

$$q_i(\omega) = \mu\big(E \mid P_i(\omega)\big)$$

**Théorème d'Aumann (1976).** Si les deux agents ont le même prior, il n'existe aucun monde où il soit de connaissance commune que $q_1 = a$ et $q_2 = b$ avec $a \neq b$.

Le carnet ne se contente pas de citer le théorème : il l'**éprouve par énumération exhaustive** sur tous les modèles à quatre mondes — les 15 partitions de $\Omega$, leurs 225 couples, et les 14 événements non triviaux. Soit 3150 modèles-événements.

La formalisation du test est le point délicat. « Ils sont en désaccord » est l'événement $D = \{\omega : q_1(\omega) \neq q_2(\omega)\}$. Ce n'est **pas** l'énoncé du théorème, qui porte sur la connaissance commune des **valeurs** : pour chaque couple $(a, b)$ avec $a \neq b$, l'événement $V_{a,b} = \{\omega : q_1(\omega) = a,\ q_2(\omega) = b\}$. Le code teste $V_{a,b}$, et la cellule suivante mesure aussi ce que donnerait le test naïf sur $D$.

In [7]:
// Modele a partitions : un monde fini, une partition par joueur, un prior commun.
// Un monde est un indice 0..n ; un evenement est un masque u32 (bit w = monde w present).

#[allow(dead_code)]
struct Modele {
    n: usize,
    partitions: Vec<Vec<u32>>,
    prior: Vec<f64>,
}

#[allow(dead_code)]
impl Modele {
    /// Verifie que chaque partition est bien une partition de Omega.
    fn valide(&self) -> bool {
        let plein = (1u32 << self.n) - 1;
        for part in &self.partitions {
            let mut union = 0u32;
            let mut total = 0u32;
            for bloc in part {
                if *bloc == 0 {
                    return false;
                }
                union |= bloc;
                total += bloc.count_ones();
            }
            if union != plein || total as usize != self.n {
                return false;
            }
        }
        true
    }

    /// Cellule d'information du joueur dans ce monde.
    fn cellule(&self, joueur: usize, monde: usize) -> u32 {
        let mut union = 0u32;
        for bloc in &self.partitions[joueur] {
            if bloc & (1u32 << monde) != 0 {
                union |= bloc;
            }
        }
        union
    }

    /// K_i(E) : les mondes ou le joueur i sait que E.
    fn sait(&self, joueur: usize, evenement: u32) -> u32 {
        let mut out = 0u32;
        for w in 0..self.n {
            if self.cellule(joueur, w) & !evenement == 0 {
                out |= 1u32 << w;
            }
        }
        out
    }
}

/// M(E) = intersection des K_i(E) sur tous les joueurs.
fn connaissance_mutuelle(modele: &Modele, ev: u32) -> u32 {
    if modele.partitions.is_empty() {
        return 0;
    }
    let mut out = modele.sait(0, ev);
    for i in 1..modele.partitions.len() {
        out &= modele.sait(i, ev);
    }
    out
}

/// CK(E) par iteration jusqu'au point fixe. Retourne (CK(E), trace de la suite).
fn connaissance_commune(modele: &Modele, ev: u32) -> (u32, Vec<u32>) {
    let mut current = ev;
    let mut trace: Vec<u32> = vec![current];
    loop {
        let nxt = connaissance_mutuelle(modele, current) & current;
        trace.push(nxt);
        if nxt == current {
            return (current, trace);
        }
        current = nxt;
    }
}

/// Enumere toutes les partitions d'un ensemble fini (restricted growth strings).
fn set_partitions(items: &[usize]) -> Vec<Vec<Vec<usize>>> {
    if items.is_empty() {
        return vec![Vec::new()];
    }
    let first = items[0];
    let rest = &items[1..];
    let mut out: Vec<Vec<Vec<usize>>> = Vec::new();
    for part in set_partitions(rest) {
        for i in 0..part.len() {
            let mut nouvelle: Vec<Vec<usize>> = Vec::new();
            for (j, bloc) in part.iter().enumerate() {
                if j == i {
                    let mut tete = vec![first];
                    tete.extend(bloc.iter().cloned());
                    nouvelle.push(tete);
                } else {
                    nouvelle.push(bloc.clone());
                }
            }
            out.push(nouvelle);
        }
        let mut nouvelle: Vec<Vec<usize>> = vec![vec![first]];
        nouvelle.extend(part.into_iter());
        out.push(nouvelle);
    }
    out
}

/// Combinaisons de taille r sur 0..n, dans l'ordre lexicographique.
fn combinations(n: usize, r: usize) -> Vec<Vec<usize>> {
    fn rec(n: usize, r: usize, debut: usize, courante: &mut Vec<usize>, out: &mut Vec<Vec<usize>>) {
        if courante.len() == r {
            out.push(courante.clone());
            return;
        }
        let reste = r - courante.len();
        let mut x = debut;
        while x < n {
            if n - x < reste {
                break;
            }
            courante.push(x);
            rec(n, r, x + 1, courante, out);
            courante.pop();
            x += 1;
        }
    }
    let mut out: Vec<Vec<usize>> = Vec::new();
    let mut courante: Vec<usize> = Vec::new();
    rec(n, r, 0, &mut courante, &mut out);
    out
}

/// q_i(omega) = mu(E | cellule du joueur en omega).
fn posterior(modele: &Modele, joueur: usize, event: u32) -> Vec<f64> {
    let mut out = vec![0.0f64; modele.n];
    for w in 0..modele.n {
        let bloc = modele.cellule(joueur, w);
        let mut masse = 0.0f64;
        for x in 0..modele.n {
            if bloc & (1u32 << x) != 0 {
                masse += modele.prior[x];
            }
        }
        let mut num = 0.0f64;
        for x in 0..modele.n {
            if bloc & event & (1u32 << x) != 0 {
                num += modele.prior[x];
            }
        }
        out[w] = num / masse;
    }
    out
}

/// Arrondi decimal a 12 chiffres, comme round(x, 12) en Python.
fn round12(x: f64) -> f64 {
    format!("{:.12}", x).parse::<f64>().unwrap_or(x)
}

/// Enumere tous les couples de partitions et tous les evenements non triviaux.
///
/// Compte (1) les paires de valeurs en desaccord examinees, (2) celles dont le
/// desaccord est de connaissance commune -- le theoreme d'Aumann en attend zero,
/// (3) ce que rendrait le test naif sur l'evenement D = {q1 != q2}.
fn balayage_aumann(worlds: &[usize], prior: &[f64]) -> (usize, usize, usize, usize, usize) {
    let n = worlds.len();
    let mut partitions: Vec<Vec<u32>> = Vec::new();
    for p in set_partitions(worlds) {
        let mut blocs: Vec<u32> = Vec::new();
        for bloc in p {
            let mut m = 0u32;
            for x in bloc {
                m |= 1u32 << x;
            }
            blocs.push(m);
        }
        partitions.push(blocs);
    }
    let mut evenements: Vec<u32> = Vec::new();
    for r in 1..n {
        for comb in combinations(n, r) {
            let mut m = 0u32;
            for x in comb {
                m |= 1u32 << x;
            }
            evenements.push(m);
        }
    }
    let mut paires_examinees = 0usize;
    let mut violations = 0usize;
    let mut test_naif = 0usize;
    for pa in &partitions {
        for pb in &partitions {
            let modele = Modele {
                n: n,
                partitions: vec![pa.clone(), pb.clone()],
                prior: prior.to_vec(),
            };
            for &ev in &evenements {
                let qa = posterior(&modele, 0, ev);
                let qb = posterior(&modele, 1, ev);
                let mut deja_vus: Vec<(f64, f64)> = Vec::new();
                for w in 0..n {
                    let cle = (round12(qa[w]), round12(qb[w]));
                    if deja_vus.contains(&cle) {
                        continue;
                    }
                    deja_vus.push(cle);
                    if qa[w] == qb[w] {
                        continue;
                    }
                    let mut evenement_valeurs = 0u32;
                    for x in 0..n {
                        if (round12(qa[x]), round12(qb[x])) == cle {
                            evenement_valeurs |= 1u32 << x;
                        }
                    }
                    paires_examinees += 1;
                    if connaissance_commune(&modele, evenement_valeurs).0 & (1u32 << w) != 0 {
                        violations += 1;
                    }
                }
                let mut d = 0u32;
                for x in 0..n {
                    if qa[x] != qb[x] {
                        d |= 1u32 << x;
                    }
                }
                let ckd = connaissance_commune(&modele, d).0;
                test_naif += (d & ckd).count_ones() as usize;
            }
        }
    }
    (partitions.len(), evenements.len(), paires_examinees, violations, test_naif)
}

let (n_part, n_ev, paires, violations, naif) = balayage_aumann(&[0, 1, 2, 3], &[0.25; 4]);
println!("partitions de 4 mondes              : {}", n_part);
println!("evenements non triviaux             : {}", n_ev);
println!("modeles-evenements balayes          : {}", n_part * n_part * n_ev);
println!("paires de valeurs en desaccord      : {}", paires);
println!("violations d'Aumann (attendu : 0)   : {}", violations);
println!("test naif sur D = {{q1 != q2}}        : {} cas de desaccord de connaissance commune", naif);

partitions de 4 mondes              : 15


evenements non triviaux             : 14


modeles-evenements balayes          : 3150


paires de valeurs en desaccord      : 5632


violations d'Aumann (attendu : 0)   : 0


test naif sur D = {q1 != q2}        : 4832 cas de desaccord de connaissance commune


### Lecture du résultat

Le test correct rend **0 violation** sur les 5632 paires de valeurs en désaccord examinées : sous prior commun, aucun couple de posteriors distincts n'est jamais de connaissance commune. Le théorème tient sur l'énumération complète, pas sur un exemple choisi.

Le test naïf, lui, rend **4832 cas** de désaccord de connaissance commune. Ce n'est pas une réfutation d'Aumann : c'est la mesure de l'écart entre deux événements différents. Que les agents sachent qu'ils **ne sont pas d'accord** n'empêche rien — ils peuvent parfaitement le savoir, et même le savoir mutuellement, dans un modèle où chacun interprète l'information différemment. Ce que le théorème interdit, c'est qu'ils sachent **sur quelles valeurs précises** ils diffèrent, parce que cette connaissance-là, jointe au prior commun, permettrait de recalculer la moyenne des deux posteriors et de forcer l'égalité.

Un carnet qui testerait $D$ conclurait « Aumann est faux » et aurait tort. La formalisation est ici tout le contenu, et le nombre 4832 est là pour qu'on s'en souvienne.

In [8]:
// Modele a partitions : un monde fini, une partition par joueur, un prior commun.
// Un monde est un indice 0..n ; un evenement est un masque u32 (bit w = monde w present).

#[allow(dead_code)]
struct Modele {
    n: usize,
    partitions: Vec<Vec<u32>>,
    prior: Vec<f64>,
}

#[allow(dead_code)]
impl Modele {
    /// Verifie que chaque partition est bien une partition de Omega.
    fn valide(&self) -> bool {
        let plein = (1u32 << self.n) - 1;
        for part in &self.partitions {
            let mut union = 0u32;
            let mut total = 0u32;
            for bloc in part {
                if *bloc == 0 {
                    return false;
                }
                union |= bloc;
                total += bloc.count_ones();
            }
            if union != plein || total as usize != self.n {
                return false;
            }
        }
        true
    }

    /// Cellule d'information du joueur dans ce monde.
    fn cellule(&self, joueur: usize, monde: usize) -> u32 {
        let mut union = 0u32;
        for bloc in &self.partitions[joueur] {
            if bloc & (1u32 << monde) != 0 {
                union |= bloc;
            }
        }
        union
    }

    /// K_i(E) : les mondes ou le joueur i sait que E.
    fn sait(&self, joueur: usize, evenement: u32) -> u32 {
        let mut out = 0u32;
        for w in 0..self.n {
            if self.cellule(joueur, w) & !evenement == 0 {
                out |= 1u32 << w;
            }
        }
        out
    }
}

/// M(E) = intersection des K_i(E) sur tous les joueurs.
fn connaissance_mutuelle(modele: &Modele, ev: u32) -> u32 {
    if modele.partitions.is_empty() {
        return 0;
    }
    let mut out = modele.sait(0, ev);
    for i in 1..modele.partitions.len() {
        out &= modele.sait(i, ev);
    }
    out
}

/// CK(E) par iteration jusqu'au point fixe. Retourne (CK(E), trace de la suite).
fn connaissance_commune(modele: &Modele, ev: u32) -> (u32, Vec<u32>) {
    let mut current = ev;
    let mut trace: Vec<u32> = vec![current];
    loop {
        let nxt = connaissance_mutuelle(modele, current) & current;
        trace.push(nxt);
        if nxt == current {
            return (current, trace);
        }
        current = nxt;
    }
}

/// q_i(omega) = mu(E | cellule du joueur en omega).
fn posterior(modele: &Modele, joueur: usize, event: u32) -> Vec<f64> {
    let mut out = vec![0.0f64; modele.n];
    for w in 0..modele.n {
        let bloc = modele.cellule(joueur, w);
        let mut masse = 0.0f64;
        for x in 0..modele.n {
            if bloc & (1u32 << x) != 0 {
                masse += modele.prior[x];
            }
        }
        let mut num = 0.0f64;
        for x in 0..modele.n {
            if bloc & event & (1u32 << x) != 0 {
                num += modele.prior[x];
            }
        }
        out[w] = num / masse;
    }
    out
}

/// Arrondi decimal a 4 chiffres, comme round(x, 4) en Python.
fn round4(x: f64) -> f64 {
    format!("{:.4}", x).parse::<f64>().unwrap_or(x)
}

fn py_list(mask: u32) -> String {
    let mut parts: Vec<String> = Vec::new();
    for w in 0..32 {
        if mask & (1u32 << w) != 0 {
            parts.push(format!("{}", w));
        }
    }
    format!("[{}]", parts.join(", "))
}

// Controle negatif : deux agents aux priors DIFFERENTS.
// Les partitions sont grossieres des deux cotes : chacun envisage tous les mondes, donc le desaccord
// sur les valeurs est de connaissance commune partout -- et c'est coherent avec Aumann,
// qui suppose le prior commun.
const PRIOR_1: [f64; 4] = [0.25, 0.25, 0.25, 0.25];
const PRIOR_2: [f64; 4] = [0.70, 0.10, 0.10, 0.10];
const E_CTRL: u32 = 0b0001;
const PLEIN: u32 = 0b1111;

let modele_1 = Modele {
    n: 4,
    partitions: vec![vec![PLEIN], vec![PLEIN]],
    prior: PRIOR_1.to_vec(),
};
let modele_2 = Modele {
    n: 4,
    partitions: vec![vec![PLEIN], vec![PLEIN]],
    prior: PRIOR_2.to_vec(),
};

let q1 = posterior(&modele_1, 0, E_CTRL);
let q2 = posterior(&modele_2, 0, E_CTRL);

// On mesure les deux joueurs du meme modele mais sous des priors differents :
// chacun croit le monde entier possible, l'ecart est donc identique partout.
let ecart: Vec<f64> = (0..4).map(|w| round4(q1[w])).collect();
println!("prior uniforme  -> q =  {}", ecart[0]);
let mut somme_biaise = 0.0f64;
for x in 0..4 {
    if PLEIN & E_CTRL & (1u32 << x) != 0 {
        somme_biaise += PRIOR_2[x];
    }
}
println!("prior biaise    -> q =  {}", round4(somme_biaise));
let mut evenement_ecart = 0u32;
for w in 0..4 {
    if (q1[w] - q2[w]).abs() > 1e-9 {
        evenement_ecart |= 1u32 << w;
    }
}
let modele_ck = Modele {
    n: 4,
    partitions: vec![vec![PLEIN], vec![PLEIN]],
    prior: PRIOR_1.to_vec(),
};
let ck_ecart = connaissance_commune(&modele_ck, evenement_ecart).0;
println!("desaccord             : {}", py_list(evenement_ecart));
println!("CK(desaccord)         : {}", py_list(ck_ecart));

prior uniforme  -> q =  0.25


prior biaise    -> q =  0.7


desaccord             : [0, 1, 2, 3]


CK(desaccord)         : [0, 1, 2, 3]


### Lecture du contrôle négatif

Avec le même prior, deux agents qui ont cette même information calculent le même posterior : pas de désaccord. Avec des priors différents, l'écart existe et il est **de connaissance commune partout** — les deux partitions sont grossières ; ici, le désaccord est l’événement Ω, qui est auto-évident.

C'est exactement ce que le théorème autorise : il ne dit pas que le désaccord est impossible, il dit que **le prior commun l'interdit**. Retirer l'hypothèse du prior commun suffit à faire apparaître le désaccord de connaissance commune. Le contrôle négatif n'est donc pas décoratif : sans lui, un carnet qui ne trouverait aucun désaccord ne saurait pas distinguer « le théorème tient » de « le modèle est dégénéré ».

### Plus loin : le monde à six états — 203 partitions, 41 209 couples

Le banc à quatre mondes balayait 15 partitions et 225 couples. Le monde à six états change l'ordre de grandeur : le nombre de Bell $B_6 = 203$ donne $203^2 = 41\,209$ couples d'informations, croisés avec les $2^6 = 64$ événements. Deux adaptations rendent le balayage exact et supportable :

- les mondes deviennent des **masques binaires** et les partitions des tuples de masques — le meet redevient un union-find sur six entiers ;
- les posteriors sont comparés en **rationnels exacts** — chaque rapport $a/b$ réduit par pgcd — pour qu'aucun arrondi flottant ne fabrique ni ne masque une égalité.

Le critère de connaissance commune devient : les deux posteriors sont constants sur la cellule du meet (chacun connaît son posterior partout dans sa cellule ; la connaissance commune exige la constance sur toute la cellule publique). Chaque cas ainsi isolé doit vérifier l'égalité — une seule égalité fausse réfuterait le théorème sur ce monde.

In [9]:
// Monde a six etats {0..5}, representes par des masques binaires.

const N: usize = 6;

/// Toutes les partitions de {0..n-1}, comme tuples de masques binaires.
fn bell_partitions(n: usize) -> Vec<Vec<u32>> {
    fn rec(i: usize, n: usize, blocs: &mut Vec<u32>, out: &mut Vec<Vec<u32>>) {
        if i == n {
            let mut tri = blocs.clone();
            tri.sort();
            out.push(tri);
            return;
        }
        let mut vus: Vec<u32> = Vec::new();
        for k in 0..blocs.len() {
            if vus.contains(&blocs[k]) {
                continue;
            }
            vus.push(blocs[k]);
            blocs[k] |= 1u32 << i;
            rec(i + 1, n, blocs, out);
            blocs[k] &= !(1u32 << i);
        }
        blocs.push(1u32 << i);
        rec(i + 1, n, blocs, out);
        blocs.pop();
    }
    let mut out: Vec<Vec<u32>> = Vec::new();
    let mut blocs: Vec<u32> = Vec::new();
    rec(0, n, &mut blocs, &mut out);
    out
}

/// Cellules du meet de deux partitions a six etats (fermeture transitive).
fn meet6(p1: &[u32], p2: &[u32]) -> Vec<u32> {
    let mut parent: Vec<usize> = (0..N).collect();

    fn find(parent: &mut Vec<usize>, x: usize) -> usize {
        let mut x = x;
        while parent[x] != x {
            parent[x] = parent[parent[x]];
            x = parent[x];
        }
        x
    }

    for part in [p1, p2] {
        for mask in part {
            let mut racine: Option<usize> = None;
            for s in 0..N {
                if mask >> s & 1 == 1 {
                    let r = find(&mut parent, s);
                    match racine {
                        None => racine = Some(r),
                        Some(root) => parent[r] = root,
                    }
                }
            }
        }
    }
    let mut racines: Vec<usize> = Vec::new();
    let mut cellules: Vec<u32> = Vec::new();
    for s in 0..N {
        let r = find(&mut parent, s);
        match racines.iter().position(|x| *x == r) {
            Some(pos) => cellules[pos] |= 1u32 << s,
            None => {
                racines.push(r);
                cellules.push(1u32 << s);
            }
        }
    }
    cellules
}

/// table[E] = poids du prior sur les mondes de E (prior en entiers).
fn table_poids(prior: &[i64]) -> Vec<i64> {
    (0..(1usize << N)).map(|e| {
        let mut s = 0i64;
        for w in 0..N {
            if e >> w & 1 == 1 {
                s += prior[w];
            }
        }
        s
    }).collect()
}

/// Cle exacte du rapport a/b, reduite par pgcd (comparaison exacte).
fn cle_exacte(a: i64, b: i64) -> (i64, i64) {
    let mut x = a.abs();
    let mut y = b.abs();
    while y != 0 {
        let t = x % y;
        x = y;
        y = t;
    }
    if x == 0 {
        return (0, 0);
    }
    (a / x, b / x)
}

/// Entier groupe par virgules, comme le format f"{n:,}" de Python.
fn py_thousands(v: u64) -> String {
    let s = v.to_string();
    let mut out = String::new();
    for (i, ch) in s.chars().enumerate() {
        if i > 0 && (s.len() - i) % 3 == 0 {
            out.push(',');
        }
        out.push(ch);
    }
    out
}

/// Balaye couples x cellules du meet x evenements. Pour chaque cas ou les
/// deux posteriors sont constants sur la cellule du meet (connaissance commune),
/// verifie l'egalite. Retourne (cas examines, violations, un exemple ou None).
fn balayage_meet(
    partitions6: &[Vec<u32>],
    prior1: &[i64],
    prior2: &[i64],
    exemple: bool,
) -> (usize, usize, Option<(Vec<u32>, Vec<u32>, u32, usize, (i64, i64), (i64, i64))>) {
    let w1 = table_poids(prior1);
    let w2 = table_poids(prior2);
    let mut cas = 0usize;
    let mut violations = 0usize;
    let mut premier: Option<(Vec<u32>, Vec<u32>, u32, usize, (i64, i64), (i64, i64))> = None;
    for p1 in partitions6 {
        for p2 in partitions6 {
            for m in meet6(p1, p2) {
                let c1: Vec<u32> = p1.iter().cloned().filter(|c| c & m != 0).collect();
                let c2: Vec<u32> = p2.iter().cloned().filter(|c| c & m != 0).collect();
                for e in 0..(1usize << N) {
                    let e = e as u32;
                    let mut q1: Vec<(i64, i64)> = Vec::new();
                    for c in &c1 {
                        let cle = cle_exacte(w1[(e & c) as usize], w1[*c as usize]);
                        if !q1.contains(&cle) {
                            q1.push(cle);
                        }
                    }
                    if q1.len() != 1 {
                        continue;
                    }
                    let mut q2: Vec<(i64, i64)> = Vec::new();
                    for c in &c2 {
                        let cle = cle_exacte(w2[(e & c) as usize], w2[*c as usize]);
                        if !q2.contains(&cle) {
                            q2.push(cle);
                        }
                    }
                    if q2.len() != 1 {
                        continue;
                    }
                    cas += 1;
                    let (a1, b1) = q1[0];
                    let (a2, b2) = q2[0];
                    if a1 * b2 != a2 * b1 {
                        violations += 1;
                        if exemple && premier.is_none() {
                            premier = Some((p1.clone(), p2.clone(), m, e as usize, (a1, b1), (a2, b2)));
                        }
                    }
                }
            }
        }
    }
    (cas, violations, premier)
}

let mut partitions6 = bell_partitions(N);
partitions6.sort();
partitions6.dedup();
println!("Monde a {} etats : {} partitions (nombre de Bell B_{}), {} couples, {} evenements",
         N, partitions6.len(), N, partitions6.len() * partitions6.len(), 1u32 << N);

println!("Balayage exhaustif (prior commun uniforme)...");
let (cas6, viol6, _) = balayage_meet(&partitions6, &[1; N], &[1; N], false);
println!("  cas de posteriors connaissance commune : {}", py_thousands(cas6 as u64));
println!("  desaccords (posteriors ck et differents) : {}", viol6);
println!();
println!("Theoreme d'Aumann sur le monde a six etats : {}",
         if viol6 == 0 { "VERIFIE (0 desaccord)" } else { "REFUTE" });

Monde a 6 etats : 203 partitions (nombre de Bell B_6), 41209 couples, 64 evenements


Balayage exhaustif (prior commun uniforme)...


  cas de posteriors connaissance commune : 1,760,900


  desaccords (posteriors ck et differents) : 0


Theoreme d'Aumann sur le monde a six etats : VERIFIE (0 desaccord)


### Lecture du résultat

$1\,760\,900$ cas de posteriors connaissance commune, et pas un seul désaccord : à l'échelle du monde à six états, l'énoncé d'Aumann tient partout où ses hypothèses tiennent. Le banc à quatre mondes (3150 modèles-événements) et celui-ci explorent le même théorème à deux échelles : le petit isole la mécanique des valeurs $V_{a,b}$ et le piège du test naïf, le grand mesure la frontière.

### La frontière du théorème, mesurée au monde à six états


In [10]:
// Monde a six etats {0..5}, representes par des masques binaires.

const N: usize = 6;

/// Toutes les partitions de {0..n-1}, comme tuples de masques binaires.
fn bell_partitions(n: usize) -> Vec<Vec<u32>> {
    fn rec(i: usize, n: usize, blocs: &mut Vec<u32>, out: &mut Vec<Vec<u32>>) {
        if i == n {
            let mut tri = blocs.clone();
            tri.sort();
            out.push(tri);
            return;
        }
        let mut vus: Vec<u32> = Vec::new();
        for k in 0..blocs.len() {
            if vus.contains(&blocs[k]) {
                continue;
            }
            vus.push(blocs[k]);
            blocs[k] |= 1u32 << i;
            rec(i + 1, n, blocs, out);
            blocs[k] &= !(1u32 << i);
        }
        blocs.push(1u32 << i);
        rec(i + 1, n, blocs, out);
        blocs.pop();
    }
    let mut out: Vec<Vec<u32>> = Vec::new();
    let mut blocs: Vec<u32> = Vec::new();
    rec(0, n, &mut blocs, &mut out);
    out
}

/// Cellules du meet de deux partitions a six etats (fermeture transitive).
fn meet6(p1: &[u32], p2: &[u32]) -> Vec<u32> {
    let mut parent: Vec<usize> = (0..N).collect();

    fn find(parent: &mut Vec<usize>, x: usize) -> usize {
        let mut x = x;
        while parent[x] != x {
            parent[x] = parent[parent[x]];
            x = parent[x];
        }
        x
    }

    for part in [p1, p2] {
        for mask in part {
            let mut racine: Option<usize> = None;
            for s in 0..N {
                if mask >> s & 1 == 1 {
                    let r = find(&mut parent, s);
                    match racine {
                        None => racine = Some(r),
                        Some(root) => parent[r] = root,
                    }
                }
            }
        }
    }
    let mut racines: Vec<usize> = Vec::new();
    let mut cellules: Vec<u32> = Vec::new();
    for s in 0..N {
        let r = find(&mut parent, s);
        match racines.iter().position(|x| *x == r) {
            Some(pos) => cellules[pos] |= 1u32 << s,
            None => {
                racines.push(r);
                cellules.push(1u32 << s);
            }
        }
    }
    cellules
}

/// table[E] = poids du prior sur les mondes de E (prior en entiers).
fn table_poids(prior: &[i64]) -> Vec<i64> {
    (0..(1usize << N)).map(|e| {
        let mut s = 0i64;
        for w in 0..N {
            if e >> w & 1 == 1 {
                s += prior[w];
            }
        }
        s
    }).collect()
}

/// Cle exacte du rapport a/b, reduite par pgcd (comparaison exacte).
fn cle_exacte(a: i64, b: i64) -> (i64, i64) {
    let mut x = a.abs();
    let mut y = b.abs();
    while y != 0 {
        let t = x % y;
        x = y;
        y = t;
    }
    if x == 0 {
        return (0, 0);
    }
    (a / x, b / x)
}

/// Entier groupe par virgules, comme le format f"{n:,}" de Python.
fn py_thousands(v: u64) -> String {
    let s = v.to_string();
    let mut out = String::new();
    for (i, ch) in s.chars().enumerate() {
        if i > 0 && (s.len() - i) % 3 == 0 {
            out.push(',');
        }
        out.push(ch);
    }
    out
}

/// Balaye couples x cellules du meet x evenements. Pour chaque cas ou les
/// deux posteriors sont constants sur la cellule du meet (connaissance commune),
/// verifie l'egalite. Retourne (cas examines, violations, un exemple ou None).
fn balayage_meet(
    partitions6: &[Vec<u32>],
    prior1: &[i64],
    prior2: &[i64],
    exemple: bool,
) -> (usize, usize, Option<(Vec<u32>, Vec<u32>, u32, usize, (i64, i64), (i64, i64))>) {
    let w1 = table_poids(prior1);
    let w2 = table_poids(prior2);
    let mut cas = 0usize;
    let mut violations = 0usize;
    let mut premier: Option<(Vec<u32>, Vec<u32>, u32, usize, (i64, i64), (i64, i64))> = None;
    for p1 in partitions6 {
        for p2 in partitions6 {
            for m in meet6(p1, p2) {
                let c1: Vec<u32> = p1.iter().cloned().filter(|c| c & m != 0).collect();
                let c2: Vec<u32> = p2.iter().cloned().filter(|c| c & m != 0).collect();
                for e in 0..(1usize << N) {
                    let e = e as u32;
                    let mut q1: Vec<(i64, i64)> = Vec::new();
                    for c in &c1 {
                        let cle = cle_exacte(w1[(e & c) as usize], w1[*c as usize]);
                        if !q1.contains(&cle) {
                            q1.push(cle);
                        }
                    }
                    if q1.len() != 1 {
                        continue;
                    }
                    let mut q2: Vec<(i64, i64)> = Vec::new();
                    for c in &c2 {
                        let cle = cle_exacte(w2[(e & c) as usize], w2[*c as usize]);
                        if !q2.contains(&cle) {
                            q2.push(cle);
                        }
                    }
                    if q2.len() != 1 {
                        continue;
                    }
                    cas += 1;
                    let (a1, b1) = q1[0];
                    let (a2, b2) = q2[0];
                    if a1 * b2 != a2 * b1 {
                        violations += 1;
                        if exemple && premier.is_none() {
                            premier = Some((p1.clone(), p2.clone(), m, e as usize, (a1, b1), (a2, b2)));
                        }
                    }
                }
            }
        }
    }
    (cas, violations, premier)
}

/// Ensemble de mondes en notation compacte, sans espace.
fn bits(m: u32) -> String {
    let mut parts: Vec<String> = Vec::new();
    for s in 0..N {
        if m >> s & 1 == 1 {
            parts.push(format!("{}", s));
        }
    }
    format!("{{{}}}", parts.join(","))
}

// Prior commun uniforme pour le joueur 1 ; prior tente pour le joueur 2.
let prior_j1 = [1i64, 1, 1, 1, 1, 1];
let prior_j2 = [1i64, 1, 1, 1, 1, 4];

let mut partitions6 = bell_partitions(N);
partitions6.sort();
partitions6.dedup();

println!("Balayage exhaustif (priors distincts : uniforme vs (1,1,1,1,1,4))...");
let (cas_n, viol_n, exemple) = balayage_meet(&partitions6, &prior_j1, &prior_j2, true);
println!("  cas de posteriors connaissance commune : {}", py_thousands(cas_n as u64));
println!("  desaccords (posteriors ck et differents) : {}", py_thousands(viol_n as u64));
println!();

let _ = match exemple {
    Some((p1, p2, m, e, (a1, b1), (a2, b2))) => {
        println!("Un desaccord connaissance-commune, en detail :");
        println!("  partition joueur 1 : {}", p1.iter().map(|c| bits(*c)).collect::<Vec<_>>().join(" | "));
        println!("  partition joueur 2 : {}", p2.iter().map(|c| bits(*c)).collect::<Vec<_>>().join(" | "));
        println!("  cellule du meet    : {}  evenement E = {}", bits(m), bits(e as u32));
        println!("  posterior joueur 1 : {}/{} = {:.4}   (prior uniforme)", a1, b1, a1 as f64 / b1 as f64);
        println!("  posterior joueur 2 : {}/{} = {:.4}   (prior tente)", a2, b2, a2 as f64 / b2 as f64);
        println!("  -> connaissance commune de posteriors DIFFERENTS : le theoreme ne s'applique pas");
    }
    None => {
        println!("Aucun desaccord trouve : controle negatif a revoir");
    }
};

Balayage exhaustif (priors distincts : uniforme vs (1,1,1,1,1,4))...


  cas de posteriors connaissance commune : 1,753,520


  desaccords (posteriors ck et differents) : 53,962


Un desaccord connaissance-commune, en detail :


  partition joueur 1 : {0} | {1} | {2} | {3} | {4,5}


  partition joueur 2 : {0} | {1} | {2} | {3} | {4,5}


  cellule du meet    : {4,5}  evenement E = {4}


  posterior joueur 1 : 1/2 = 0.5000   (prior uniforme)


  posterior joueur 2 : 1/5 = 0.2000   (prior tente)


  -> connaissance commune de posteriors DIFFERENTS : le theoreme ne s'applique pas


### Lecture du contrôle négatif

L'hypothèse « same priors » n'était pas décorative — le contrôle négatif à quatre mondes le laissait deviner sur des partitions triviales. Ici, le même balayage est rejoué, mais le joueur 2 part d'un prior tenté $(1,1,1,1,1,4)$ : le critère de constance sur le meet isole encore des cas de posteriors connaissance commune, et cette fois l'égalité a cédé — la sortie montre les deux visages, le décompte puis un cas concret.

$53\,962$ désaccords de connaissance commune sur $1\,753\,520$ cas : à peine 3 % des cas isolés, mais ils existent — et le cas affiché montre la mécanique. Les deux joueurs ont des posteriors constants sur la cellule du meet ($1/2$ partout pour l'un, $1/5$ partout pour l'autre), donc de connaissance commune, et néanmoins différents : chacun conditionne juste, mais sur des croyances antérieures incompatibles. La frontière du théorème n'est pas l'information — elle est le prior.

### Exercice 3 — trois agents, même théorème

Le théorème d'Aumann s'énonce pour deux agents ; avec trois agents qui partagent un prior, la conclusion tient encore. Vérifiez-le sur un seul modèle, sans balayage exhaustif : retournez `Some((modele, E))` : construisez trois partitions de $\{0,1,2,3\}$ et un événement $E$, calculez les trois posteriors, et cherchez un monde où les trois valeurs sont deux à deux distinctes. Montrez que l'événement « ces trois valeurs précises » n'est jamais de connaissance commune.

Indice : `connaissance_commune` prend un modèle ; `Modele` accepte autant de partitions que vous voulez dans son champ `partitions`. Cherchez d'abord un désaccord entre trois agents, puis testez sa connaissance commune — le résultat intéressant est que le désaccord existe, abondamment, mais qu'il n'est jamais su.

In [11]:
// Modele a partitions : un monde fini, une partition par joueur, un prior commun.
// Un monde est un indice 0..n ; un evenement est un masque u32 (bit w = monde w present).

#[allow(dead_code)]
struct Modele {
    n: usize,
    partitions: Vec<Vec<u32>>,
    prior: Vec<f64>,
}

#[allow(dead_code)]
impl Modele {
    /// Verifie que chaque partition est bien une partition de Omega.
    fn valide(&self) -> bool {
        let plein = (1u32 << self.n) - 1;
        for part in &self.partitions {
            let mut union = 0u32;
            let mut total = 0u32;
            for bloc in part {
                if *bloc == 0 {
                    return false;
                }
                union |= bloc;
                total += bloc.count_ones();
            }
            if union != plein || total as usize != self.n {
                return false;
            }
        }
        true
    }

    /// Cellule d'information du joueur dans ce monde.
    fn cellule(&self, joueur: usize, monde: usize) -> u32 {
        let mut union = 0u32;
        for bloc in &self.partitions[joueur] {
            if bloc & (1u32 << monde) != 0 {
                union |= bloc;
            }
        }
        union
    }

    /// K_i(E) : les mondes ou le joueur i sait que E.
    fn sait(&self, joueur: usize, evenement: u32) -> u32 {
        let mut out = 0u32;
        for w in 0..self.n {
            if self.cellule(joueur, w) & !evenement == 0 {
                out |= 1u32 << w;
            }
        }
        out
    }
}

/// M(E) = intersection des K_i(E) sur tous les joueurs.
fn connaissance_mutuelle(modele: &Modele, ev: u32) -> u32 {
    if modele.partitions.is_empty() {
        return 0;
    }
    let mut out = modele.sait(0, ev);
    for i in 1..modele.partitions.len() {
        out &= modele.sait(i, ev);
    }
    out
}

/// CK(E) par iteration jusqu'au point fixe. Retourne (CK(E), trace de la suite).
fn connaissance_commune(modele: &Modele, ev: u32) -> (u32, Vec<u32>) {
    let mut current = ev;
    let mut trace: Vec<u32> = vec![current];
    loop {
        let nxt = connaissance_mutuelle(modele, current) & current;
        trace.push(nxt);
        if nxt == current {
            return (current, trace);
        }
        current = nxt;
    }
}

/// q_i(omega) = mu(E | cellule du joueur en omega).
fn posterior(modele: &Modele, joueur: usize, event: u32) -> Vec<f64> {
    let mut out = vec![0.0f64; modele.n];
    for w in 0..modele.n {
        let bloc = modele.cellule(joueur, w);
        let mut masse = 0.0f64;
        for x in 0..modele.n {
            if bloc & (1u32 << x) != 0 {
                masse += modele.prior[x];
            }
        }
        let mut num = 0.0f64;
        for x in 0..modele.n {
            if bloc & event & (1u32 << x) != 0 {
                num += modele.prior[x];
            }
        }
        out[w] = num / masse;
    }
    out
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

/// Cherche un desaccord entre trois agents sous prior commun, puis mesure sa CK.
///
/// Retourne Some((modele, E)).
#[allow(dead_code)]
fn trois_agents_verification() -> Option<(Modele, u32)> {
    // TODO etudiant : construire un Modele a trois partitions sur Omega,
    // calculer posterior(modele, i, E) pour i = 0, 1, 2,
    // trouver un monde ou les trois valeurs sont distinctes,
    // puis appeler connaissance_commune sur l'evenement des valeurs exactes.
    println!("Exercice a completer : trois_agents_verification");
    None
}

println!("Exercice a completer : trois agents sous prior commun.");

/// Garde : aucun desaccord entre trois agents n'est de connaissance commune.
#[allow(dead_code)]
fn valider_trois_agents(resultat: &Option<(Modele, u32)>) -> bool {
    match resultat {
        None => false,
        Some((modele, evenement)) => {
            if modele.n != 4 || modele.partitions.len() != 3 || !modele.valide() {
                return false;
            }
            if modele.prior.len() != 4
                || modele.prior.iter().any(|p| !(*p > 0.0 && p.is_finite()))
                || (modele.prior.iter().sum::<f64>() - 1.0).abs() >= 1e-9 {
                return false;
            }
            let plein = (1u32 << modele.n) - 1;
            if *evenement == 0 || *evenement == plein {
                return false;
            }
            let qs: Vec<Vec<f64>> = (0..3).map(|i| posterior(modele, i, *evenement)).collect();
            for w in 0..modele.n {
                let valeurs = [qs[0][w], qs[1][w], qs[2][w]];
                if valeurs[0] == valeurs[1] || valeurs[0] == valeurs[2] || valeurs[1] == valeurs[2] {
                    continue;
                }
                let mut ev = 0u32;
                for x in 0..modele.n {
                    if qs[0][x] == valeurs[0] && qs[1][x] == valeurs[1] && qs[2][x] == valeurs[2] {
                        ev |= 1u32 << x;
                    }
                }
                if connaissance_commune(modele, ev).0 == 0 {
                    return true;
                }
            }
            false
        }
    }
}

let solution = trois_agents_verification();
let _ = if solution.is_some() {
    println!("Contrat verifie : {}", py_bool(valider_trois_agents(&solution)));
};

Exercice a completer : trois agents sous prior commun.


Exercice a completer : trois_agents_verification


## 4. Ce que l'épistémique change pour l'élimination itérée

Reste à relier tout cela au palier 02. La rationalité d'un joueur est un événement du modèle : au monde $\omega$, le joueur $i$ est rationnel si sa stratégie maximise son espérance conditionnelle à sa cellule d'information. L'hypothèse **CKR** — *common knowledge of rationality* — est alors $CK(R)$ avec $R = \bigcap_i R_i$, et elle a une conséquence calculable sur les stratégies.

Sur un jeu fini, l'élimination itérée des stratégies strictement dominées (IESDS) illustre une conséquence de la connaissance commune de la rationalité sous les hypothèses épistémiques appropriées : chaque tour correspond intuitivement à un niveau supplémentaire de raisonnement. Le code ci-dessous fait tourner l'élimination sur un jeu $3 \times 2$ et affiche l'ensemble résiduel — le lien avec les sections précédentes est conceptuel : ce calcul seul ne construit pas un modèle épistémique de rationalité.

In [12]:
/// Un jeu bipartite : lignes (joueur ligne) contre colonnes (joueur colonne).
#[allow(dead_code)]
struct Jeu {
    lignes: Vec<&'static str>,
    colonnes: Vec<&'static str>,
    gains: Vec<Vec<(i64, i64)>>,
}

#[allow(dead_code)]
impl Jeu {
    /// Gain du couple (ligne, colonne), dans l'ordre des insertions.
    fn gain(&self, l: &str, c: &str) -> (i64, i64) {
        let li = self.lignes.iter().position(|x| *x == l).unwrap_or(0);
        let ci = self.colonnes.iter().position(|x| *x == c).unwrap_or(0);
        self.gains[li][ci]
    }
}

/// IESDS : retire a chaque tour les strategies strictement dominees des deux joueurs.
#[allow(dead_code)]
fn eliminer_dominees(jeu: &Jeu) -> (Vec<&'static str>, Vec<&'static str>, Vec<(Vec<&'static str>, Vec<&'static str>)>) {
    let mut lignes = jeu.lignes.clone();
    let mut colonnes = jeu.colonnes.clone();
    let mut tours: Vec<(Vec<&'static str>, Vec<&'static str>)> = Vec::new();
    loop {
        let mut retirer_l: Vec<&'static str> = Vec::new();
        for l in &lignes {
            for l2 in &lignes {
                if l2 != l && colonnes.iter().all(|c| jeu.gain(l2, c).0 > jeu.gain(l, c).0) {
                    retirer_l.push(*l);
                    break;
                }
            }
        }
        let mut retirer_c: Vec<&'static str> = Vec::new();
        for c in &colonnes {
            for c2 in &colonnes {
                if c2 != c && lignes.iter().all(|l| jeu.gain(l, c2).1 > jeu.gain(l, c).1) {
                    retirer_c.push(*c);
                    break;
                }
            }
        }
        if retirer_l.is_empty() && retirer_c.is_empty() {
            return (lignes, colonnes, tours);
        }
        lignes.retain(|l| !retirer_l.contains(l));
        colonnes.retain(|c| !retirer_c.contains(c));
        tours.push((retirer_l, retirer_c));
    }
}

/// Liste de chaines au format repr de Python.
fn py_str_list(v: &[&str]) -> String {
    let parts: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", parts.join(", "))
}

/// Liste de tuples de listes de chaines au format repr de Python.
fn py_tours(tours: &[(Vec<&'static str>, Vec<&'static str>)]) -> String {
    let parts: Vec<String> = tours.iter().map(|(l, c)| format!("({}, {})", py_str_list(l), py_str_list(c))).collect();
    format!("[{}]", parts.join(", "))
}

let jeu = Jeu {
    lignes: vec!["T", "M", "B"],
    colonnes: vec!["L", "R"],
    gains: vec![
        vec![(3, 3), (1, 1)],
        vec![(2, 2), (1, 1)],
        vec![(0, 0), (0, 0)],
    ],
};

let (restant_l, restant_c, historique) = eliminer_dominees(&jeu);
println!("tours d'elimination : {}", py_tours(&historique));
println!("lignes survivantes  : {}", py_str_list(&restant_l));
println!("colonnes survivantes: {}", py_str_list(&restant_c));
println!("nombre de tours     : {}", historique.len());

tours d'elimination : [(['B'], []), ([], ['R']), (['M'], [])]


lignes survivantes  : ['T']


colonnes survivantes: ['L']


nombre de tours     : 3


### Lecture du résultat

L'élimination retire `['B']`, puis `['R']`, puis `['M']` : trois tours, et l'ensemble survivant est le singleton `(['T'], ['L'])`. La cascade est instructive. Au premier tour, `B` est strictement dominée par `M` (2 > 0 et 1 > 0). Au deuxième, `R` devient strictement dominée par `L` — mais elle ne l'était **pas** avant, parce que sur la ligne `B` les deux colonnes rapportaient 0 : la dominance n'apparaît qu'après une première élimination. Au troisième tour, `M` tombe face à `T`. C'est cette dépendance en chaîne qui fait de l'élimination un calcul **itératif** et non une simple passe, et c'est elle que la connaissance commune de la rationalité autorise : chaque tour suppose que l'adversaire a déjà raisonné le tour précédent.

Ce qui compte pour ce carnet n'est pas le résultat du jeu mais la **forme** du calcul : chaque tour d'élimination suppose que le joueur sait que son adversaire est rationnel, donc retire une stratégie, ce qui rend une nouvelle stratégie dominée au tour suivant. C'est la suite $M^n$ de la section 2 appliquée à l'événement de rationalité. La connaissance commune n'est donc pas un ornement philosophique du palier 02 : c'est ce qui donne un sens à « éliminer itérativement » plutôt qu'à « éliminer une fois ».

## Synthèse

Trois choses ont été **calculées** ici, et non citées :

| Résultat | Comment il est obtenu | Ce que le carnet ajoute |
|---|---|---|
| Les axiomes S5 tiennent sur un modèle à partitions | énumération exhaustive : 16 événements, 256 paires, six propriétés | la connaissance est dérivée de l'information, pas postulée |
| La connaissance mutuelle peut tenir alors que la connaissance commune est vide | itération jusqu'au point fixe, avec la suite des itérés affichée | la distinction n'est pas graduelle, elle est de nature |
| La connaissance commune se lit sur la cellule du meet | fermeture transitive des partitions, comparaison des deux définitions couple par couple | 64 couples (événement, monde), 0 contre-exemple |
| Sous prior commun, un désaccord de posteriors n'est jamais de connaissance commune | énumération exhaustive : 15 partitions, 225 couples, 14 événements | 5632 paires examinées, 0 violation — et 4832 faux contre-exemples si l'on teste la mauvaise formalisation ; au monde à six états : $1\,760\,900$ cas de posteriors connaissance commune, 0 désaccord, $53\,962$ dès que les priors divergent |

## Suite

Les chapitres suivants de Bonanno (10 à 12 : équilibre séquentiel faible, équilibre séquentiel, équilibre bayésien parfait) ne sont pas traités ici. Le palier 12 présente l'équilibre bayésien parfait en application, sans le dériver de la machinerie épistémique construite ici. Ce sont ces chapitres qui l'utilisent : les **croyances hors équilibre** sont exactement des cellules d'information auxquelles on affecte un posterior, et la cohérence d'une évaluation est une condition sur la façon dont ces posteriors se déduisent les uns des autres.

## Références

- Bonanno, G. (2015). *Game Theory* — ch. 7 *Knowledge and Common Knowledge*, ch. 8 *Beliefs*, ch. 9 *Common Knowledge of Rationality*. Licence CC BY-NC-ND 4.0 (page de copyright du PDF archivé). Copie archivée au gisement bibliographique (dossier `GameTheory`).
- Aumann, R. J. (1976). « Agreeing to Disagree ». *The Annals of Statistics* 4(6), 1236-1239 — l'énoncé du théorème et sa démonstration par martingales. Copie archivée au gisement bibliographique (dossier `GameTheory`) ; énoncé original lu p. 1236.
- Geanakoplos, J. (1992). « Common Knowledge ». *Journal of Economic Perspectives* 6(4), 53-82 — la revue de référence sur les modèles à partitions et leurs limites.
- Halpern, J. Y. & Moses, Y. (1992). « A Guide to Completeness and Complexity for Modal Logics of Knowledge and Belief ». *Artificial Intelligence* 54, 319-379 — la variante probabiliste de S5 utilisée ici.

## À retenir
Les partitions définissent la connaissance ; le meet permet de vérifier la connaissance commune. Les priors distincts montrent la portée de l’hypothèse d’Aumann.

## Vérifiez votre compréhension

1. Pourquoi un désaccord connu ne contredit-il pas Aumann ?

<details>
<summary>Éléments de réponse</summary>

1. Le théorème porte sur la connaissance commune des valeurs précises des posteriors.

</details>

## Pour aller plus loin
[Série GameTheory](README.md) et Aumann (1976), Agreeing to Disagree.